In [33]:
import numpy as np
import pandas as pd
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.manifold import MDS
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances
from scipy.spatial.distance import pdist, squareform


In [34]:
df = pd.read_csv('Sep0_hellinger.csv', sep=',')

In [35]:
# Keep only numeric columns
numeric_df = df.select_dtypes(include=[np.number])

In [36]:
# Define distances

#NB distance
def NB_distance(x, y):
    numerator = np.sum(np.abs(x - y) * (np.abs(x) + np.abs(y)))
    denominator = np.sum(x**2 + y**2)
    return (numerator / denominator)*(0.5) if denominator != 0 else 0

In [37]:
# Bray-Curtis distance
def bray_curtis_distance(x, y):
    num = np.sum(np.abs(x - y))
    denom = np.sum(x + y)
    return num / denom if denom != 0 else 0

In [38]:
# Kulczynski distance
def kulczynski_distance(x, y):
    min_sum = np.sum(np.minimum(x, y))
    sum_x = np.sum(x)
    sum_y = np.sum(y)
    if sum_x == 0 or sum_y == 0:
        return 1  # Max dissimilarity
    return 1 - 0.5 * ((min_sum / sum_x) + (min_sum / sum_y))

In [39]:
# Chord distance
def chord_distance(x, y):
    x_norm = x / np.sqrt(np.sum(x**2)) if np.sum(x**2) != 0 else x
    y_norm = y / np.sqrt(np.sum(y**2)) if np.sum(y**2) != 0 else y
    return np.linalg.norm(x_norm - y_norm)

In [40]:
# Chi-square distance
def chi_square_distance(x, y):
    # Oransal bolluk (pik)
    x_prop = x / np.sum(x) if np.sum(x) != 0 else np.zeros_like(x)
    y_prop = y / np.sum(y) if np.sum(y) != 0 else np.zeros_like(y)
    mean_prop = (x_prop + y_prop) / 2

    with np.errstate(divide='ignore', invalid='ignore'):
        dist = np.where(mean_prop == 0, 0, ((x_prop - y_prop) ** 2) / mean_prop)

    return np.sqrt(np.sum(dist))

In [ ]:
#Dunn index
def dunn_index(X, labels):
    """
    Dunn Index hesaplar.
    X: veri noktaları (numpy array veya pandas dataframe)
    labels: küme etiketleri (ör: [0,0,1,1,2,2])
    """
    distances = pairwise_distances(X)  
    clusters = np.unique(labels)

    # intra-cluster diameters
    intra_dists = []
    for c in clusters:
        indices = np.where(labels == c)[0]
        if len(indices) > 1:
            intra_dists.append(np.max(distances[np.ix_(indices, indices)]))
        else:
            intra_dists.append(0)  # tek nokta varsa çap = 0
    max_intra = np.max(intra_dists)

    # Minimum distances between pairs of clusters
    inter_dists = []
    for i in range(len(clusters)):
        for j in range(i+1, len(clusters)):
            idx_i = np.where(labels == clusters[i])[0]
            idx_j = np.where(labels == clusters[j])[0]
            inter_dists.append(np.min(distances[np.ix_(idx_i, idx_j)]))
    min_inter = np.min(inter_dists)

    # Dunn index
    return min_inter / max_intra if max_intra > 0 else 0


In [ ]:
#Since the artificial data consisted of 3 latent habitats, the cluster count was taken as 3.
# Helper function to compute clustering metrics
def clustering_metrics_kmeans(distance_func, name):
    # Calculate covariance matrix for Mahalanobis distance if needed
    S = None
    if name == "mahalanobis":
      S = np.cov(numeric_df.values.T) # Transpose to get covariance between features
      dist_matrix = squareform(pdist(numeric_df.values, metric=distance_func, S=S))
    else:
      dist_matrix = squareform(pdist(numeric_df.values, metric=distance_func))

    coords = MDS(n_components=2, dissimilarity='precomputed', random_state=42).fit_transform(dist_matrix)
    labels = KMeans(n_clusters=3, random_state=42).fit_predict(coords)
    silhouette = silhouette_score(coords, labels)
    db = davies_bouldin_score(coords, labels)
    ch = calinski_harabasz_score(coords, labels)
    dn  = dunn_index(coords, labels)  
    return {
        "Method": name,
        "Silhouette": silhouette,
        "Davies-Bouldin": db,
        "Calinski-Harabasz": ch,
        "Dunn": dn
    }

In [50]:
# Run the analysis
# Drop rows with any NaN values from numeric_df
numeric_df = numeric_df.dropna()

# Run the analysis
results = [
    clustering_metrics_kmeans(NB_distance, "NB"),
    clustering_metrics_kmeans(bray_curtis_distance, "Bray-Curtis"),
    clustering_metrics_kmeans(kulczynski_distance, "Kulczynski"),
    clustering_metrics_kmeans(chord_distance, "Chord"),
    clustering_metrics_kmeans(chi_square_distance, "Chi-square")
]
  


c:\Users\nursu\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\manifold\_mds.py:677: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9.
  warnings.warn(
c:\Users\nursu\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\manifold\_mds.py:677: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9.
  warnings.warn(
c:\Users\nursu\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\manifold\_mds.py:677: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9.
  warnings.warn(
c:\Users\nursu\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\manifold\_mds.py:677: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9.
  warnings.warn(
c:\Users\nursu\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\manifold\_mds.py:677: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9.
  warnings.warn(


In [51]:
results_df = pd.DataFrame(results)

from IPython.display import display
display(results_df)

,Method,Silhouette,Davies-Bouldin,Calinski-Harabasz,Dunn
0,NB,0.598652,0.512670,435.024662,0.039266
1,Bray-Curtis,0.571052,0.553461,377.030784,0.034492
2,Kulczynski,0.575024,0.551290,391.321907,0.035415
3,Chord,0.568086,0.562511,394.546991,0.045910
4,Chi-square,0.523321,0.622267,321.050254,0.039043
